### PathVQA Data Preparation

This code will prepare the PathVQA data for evaluation. We need to evaluate PathVQA as following:

Close-Ended: 50 \
Open-Ended: 550
- What
- Where
- How
- When
- Why

Since we want only the questions/answers displaying sensitive knowledge, we will only consider the data samples for which we have the length of both the answers and questions more than 5 words. If we are not able to find the respective number of samples, we can consider those data samples for which we have anwers or questions having more than 5 words.

In [1]:
import sys
from PIL import Image
import pandas as pd
import os
import pickle
from collections import Counter
import numpy as np
import shutil
import random

In [2]:
random_state = 42
np.random.seed(random_state)
random.seed(random_state)

#### Defining the paths for PVQA dataset

In [3]:
pvqa_data_path = "/data/mn27889/path-open-data/pathvqa-histopathology"
pvqa_images = os.path.join(pvqa_data_path, "images")
pvqa_qas = os.path.join(pvqa_data_path, "qas")

#### Considering images/qas from `train` subset

In [4]:
pvqa_subset_train = "train"
pvqa_images_subset_path_train = os.path.join(pvqa_images, pvqa_subset_train)
pvqa_qas_subset_path_train = os.path.join(pvqa_qas, pvqa_subset_train)

Reading the QAS

In [5]:
file_name = "train_qa.pkl"
qas_file_path_train = os.path.join(pvqa_qas_subset_path_train, file_name)
with open(qas_file_path_train, 'rb') as file:
    pvqa_qas_subset_train = pickle.load(file)

print('Total Train Samples:',len(pvqa_qas_subset_train))

Total Train Samples: 5395


#### Considering images/qas from `test` subset

In [6]:
pvqa_subset_test = "test"
pvqa_images_subset_path_test = os.path.join(pvqa_images, pvqa_subset_test)
pvqa_qas_subset_path_test = os.path.join(pvqa_qas, pvqa_subset_test)

Reading the QAS

In [7]:
file_name = "test_qa.pkl"
qas_file_path_test = os.path.join(pvqa_qas_subset_path_test, file_name)
with open(qas_file_path_test, 'rb') as file:
    pvqa_qas_subset_test = pickle.load(file)

print('Total Test Samples:',len(pvqa_qas_subset_test))

Total Test Samples: 1885


#### Considering images/qas from `val` subset

In [8]:
pvqa_subset_val = "val"
pvqa_images_subset_path_val = os.path.join(pvqa_images, pvqa_subset_val)
pvqa_qas_subset_path_val = os.path.join(pvqa_qas, pvqa_subset_val)

Reading the QAS

In [9]:
file_name = "val_qa.pkl"
qas_file_path_val = os.path.join(pvqa_qas_subset_path_val, file_name)
with open(qas_file_path_val, 'rb') as file:
    pvqa_qas_subset_val = pickle.load(file)

print('Total Val Samples:',len(pvqa_qas_subset_val))

Total Val Samples: 1712


### Combining the train, test and val datasets in one merged dataset

In [10]:
pvqa_qas_subset = pvqa_qas_subset_train + pvqa_qas_subset_test + pvqa_qas_subset_val
len(pvqa_qas_subset)

8992

In [11]:
pvqa_qas_subset[0:5]

[{'image': 'train_0422',
  'question': 'Where are liver stem cells (oval cells) located?',
  'answer': 'in the canals of hering'},
 {'image': 'train_0422',
  'question': 'What are stained here with an immunohistochemical stain for cytokeratin 7?',
  'answer': 'bile duct cells and canals of hering'},
 {'image': 'train_0422',
  'question': 'What are bile duct cells and canals of Hering stained here with for cytokeratin 7?',
  'answer': 'an immunohistochemical stain'},
 {'image': 'train_0422',
  'question': 'Are bile duct cells and canals of Hering stained here with an immunohistochemical stain for cytokeratin 7?',
  'answer': 'yes'},
 {'image': 'train_0227',
  'question': 'What is called the Prussian blue reaction?',
  'answer': 'a special staining process'}]

#### Differentiating the open-ended and close-ended questions

In [12]:
pvqa_qas_close_ended = [sample for sample in pvqa_qas_subset if sample['answer'].lower() == 'yes' or sample['answer'].lower() == 'no']
len(pvqa_qas_close_ended)

4630

In [13]:
pvqa_qas_open_ended = [sample for sample in pvqa_qas_subset if sample not in pvqa_qas_close_ended]
len(pvqa_qas_open_ended)

4362

In [14]:
assert len(pvqa_qas_open_ended) + len(pvqa_qas_close_ended) == len(pvqa_qas_subset)

#### Finding the samples for which question and answer are equal to or more than 5 words.
- For close-ended questions, only check question text since answer would be 'yes' or 'no'
- For open-ended questions, check the both the question and answer

In [15]:
valid_close_ended_samples = [sample for sample in pvqa_qas_close_ended if len(sample['question'].split()) >= 5]
len(valid_close_ended_samples)

3005

In [16]:
valid_open_ended_samples = [sample for sample in pvqa_qas_open_ended if len(sample['question'].split()) >= 5 and len(sample['answer'].split()) >= 5]
len(valid_open_ended_samples)

612

#### From open-ended questions, separating the questions starting with the following words:
- What
- Where
- How
- When
- Why

In [17]:
first_word = [sample['question'].split()[0].lower() for sample in valid_open_ended_samples]
counts = Counter(first_word)
print(counts)

Counter({'what': 528, 'how': 56, 'why': 14, 'where': 13, 'when': 1})


In [18]:
what_question_samples = [sample for sample in valid_open_ended_samples if sample['question'].lower().startswith('what')]
how_question_samples = [sample for sample in valid_open_ended_samples if sample['question'].lower().startswith('how')]
why_question_samples = [sample for sample in valid_open_ended_samples if sample['question'].lower().startswith('why')]
where_question_samples = [sample for sample in valid_open_ended_samples if sample['question'].lower().startswith('where')]
when_question_samples = [sample for sample in valid_open_ended_samples if sample['question'].lower().startswith('when')]
print('Total What Questions:', len(what_question_samples))
print('Total How Questions:', len(how_question_samples))
print('Total Why Questions:', len(why_question_samples))
print('Total Where Questions:', len(where_question_samples))
print('Total When Questions:', len(when_question_samples))

Total What Questions: 528
Total How Questions: 56
Total Why Questions: 14
Total Where Questions: 13
Total When Questions: 1


### Compiling the data into a single dataframe

Selecting the Top 50 samples of Close-Ended

In [19]:
pvqa_eval_data_close_ended = pd.DataFrame(columns=['image_path', 'question', 'answer', 'question_type'])

for sample in valid_close_ended_samples[0:50]:
    if 'train' in sample['image']:
        pvqa_images_subset_path = pvqa_images_subset_path_train
    elif 'test' in sample['image']:
        pvqa_images_subset_path = pvqa_images_subset_path_test
    else:
        pvqa_images_subset_path = pvqa_images_subset_path_val
        
    image_path = os.path.join(pvqa_images_subset_path, sample['image'] + '.jpg')
    question = sample['question']
    answer = sample['answer']
    question_type = 'close-ended'
    pvqa_eval_data_close_ended.loc[len(pvqa_eval_data_close_ended)] = [image_path, question, answer, question_type]

pvqa_eval_data_close_ended.head()

,image_path,question,answer,question_type
0,/data/mn27889/path-open-data/pathvqa-histopath...,Are bile duct cells and canals of Hering stain...,yes,close-ended
1,/data/mn27889/path-open-data/pathvqa-histopath...,Are iron deposits shown by a special staining ...,yes,close-ended
2,/data/mn27889/path-open-data/pathvqa-histopath...,Do the photomicrographs show an inflammatory r...,yes,close-ended
3,/data/mn27889/path-open-data/pathvqa-histopath...,Do apoptotic cells in colonic epithelium show ...,no,close-ended
4,/data/mn27889/path-open-data/pathvqa-histopath...,Do the photomicrographs show an inflammatory r...,yes,close-ended


Selecting all the samples of What Open-Ended

In [20]:
pvqa_eval_data_open_what = pd.DataFrame(columns=['image_path', 'question', 'answer', 'question_type'])

for sample in what_question_samples:
    if 'train' in sample['image']:
        pvqa_images_subset_path = pvqa_images_subset_path_train
    elif 'test' in sample['image']:
        pvqa_images_subset_path = pvqa_images_subset_path_test
    else:
        pvqa_images_subset_path = pvqa_images_subset_path_val

    image_path = os.path.join(pvqa_images_subset_path, sample['image'] + '.jpg')
    question = sample['question']
    answer = sample['answer']
    question_type = 'open-what'
    pvqa_eval_data_open_what.loc[len(pvqa_eval_data_open_what)] = [image_path, question, answer, question_type]

pvqa_eval_data_open_what.head()

,image_path,question,answer,question_type
0,/data/mn27889/path-open-data/pathvqa-histopath...,What are stained here with an immunohistochemi...,bile duct cells and canals of hering,open-what
1,/data/mn27889/path-open-data/pathvqa-histopath...,What is the late-phase reaction characterized by?,an inflammatory infiltrate rich in eosinophils,open-what
2,/data/mn27889/path-open-data/pathvqa-histopath...,What is manifested by inflammatory cells in th...,acute cellular rejection of a kidney graft,open-what
3,/data/mn27889/path-open-data/pathvqa-histopath...,What is acute cellular rejection of a kidney g...,inflammatory cells in the inter-stitium and be...,open-what
4,/data/mn27889/path-open-data/pathvqa-histopath...,What reveals pink-red deposits of amyloid in t...,a section liver stained with congo,open-what


Selecting all samples of How Open-Ended

In [21]:
pvqa_eval_data_open_how = pd.DataFrame(columns=['image_path', 'question', 'answer', 'question_type'])

for sample in how_question_samples:
    if 'train' in sample['image']:
        pvqa_images_subset_path = pvqa_images_subset_path_train
    elif 'test' in sample['image']:
        pvqa_images_subset_path = pvqa_images_subset_path_test
    else:
        pvqa_images_subset_path = pvqa_images_subset_path_val

    image_path = os.path.join(pvqa_images_subset_path, sample['image'] + '.jpg')
    question = sample['question']
    answer = sample['answer']
    question_type = 'open-how'
    pvqa_eval_data_open_how.loc[len(pvqa_eval_data_open_how)] = [image_path, question, answer, question_type]

pvqa_eval_data_open_how.head()

,image_path,question,answer,question_type
0,/data/mn27889/path-open-data/pathvqa-histopath...,How is acute cellular rejection of a kidney gr...,by inflammatory cells in the inter-stitium and...,open-how
1,/data/mn27889/path-open-data/pathvqa-histopath...,How is vascular changes and fibrosis of saliva...,by radiation therapy of the neck region,open-how
2,/data/mn27889/path-open-data/pathvqa-histopath...,How is vascular changes and fibrosis of saliva...,by radiation therapy of the neck region,open-how
3,/data/mn27889/path-open-data/pathvqa-histopath...,How is vascular changes and fibrosis of saliva...,by radiation therapy of the neck region,open-how
4,/data/mn27889/path-open-data/pathvqa-histopath...,How is acute endocarditis caused?,by staphylococcus aureus on a congenitally bic...,open-how


Selecting all samples of Why Open-Ended

In [22]:
pvqa_eval_data_open_why = pd.DataFrame(columns=['image_path', 'question', 'answer', 'question_type'])

for sample in why_question_samples:
    if 'train' in sample['image']:
        pvqa_images_subset_path = pvqa_images_subset_path_train
    elif 'test' in sample['image']:
        pvqa_images_subset_path = pvqa_images_subset_path_test
    else:
        pvqa_images_subset_path = pvqa_images_subset_path_val

    image_path = os.path.join(pvqa_images_subset_path, sample['image'] + '.jpg')
    question = sample['question']
    answer = sample['answer']
    question_type = 'open-why'
    pvqa_eval_data_open_why.loc[len(pvqa_eval_data_open_why)] = [image_path, question, answer, question_type]

pvqa_eval_data_open_why.head()

,image_path,question,answer,question_type
0,/data/mn27889/path-open-data/pathvqa-histopath...,Why are the alveolar septa thickened?,due to congested capillaries and neutrophilic ...,open-why
1,/data/mn27889/path-open-data/pathvqa-histopath...,Why does this image show that cut surface both...,due to mumps have no history at time,open-why
2,/data/mn27889/path-open-data/pathvqa-histopath...,"Why does this image show brain, infarct?",due to ruptured saccular aneurysm and thrombos...,open-why
3,/data/mn27889/path-open-data/pathvqa-histopath...,"Why does this image show brain, infarct?",due to ruptured saccular aneurysm and thrombos...,open-why
4,/data/mn27889/path-open-data/pathvqa-histopath...,Why does this image show spinal cord injury?,due to vertebral column trauma,open-why


Selecting all samples of Where Open-Ended

In [23]:
pvqa_eval_data_open_where = pd.DataFrame(columns=['image_path', 'question', 'answer', 'question_type'])

for sample in where_question_samples:
    if 'train' in sample['image']:
        pvqa_images_subset_path = pvqa_images_subset_path_train
    elif 'test' in sample['image']:
        pvqa_images_subset_path = pvqa_images_subset_path_test
    else:
        pvqa_images_subset_path = pvqa_images_subset_path_val

    image_path = os.path.join(pvqa_images_subset_path, sample['image'] + '.jpg')
    question = sample['question']
    answer = sample['answer']
    question_type = 'open-where'
    pvqa_eval_data_open_where.loc[len(pvqa_eval_data_open_where)] = [image_path, question, answer, question_type]

pvqa_eval_data_open_where.head()

,image_path,question,answer,question_type
0,/data/mn27889/path-open-data/pathvqa-histopath...,Where are liver stem cells (oval cells) located?,in the canals of hering,open-where
1,/data/mn27889/path-open-data/pathvqa-histopath...,Where does hepatocellular iron appear blue?,in the prussian blue-stained section,open-where
2,/data/mn27889/path-open-data/pathvqa-histopath...,Where are scattered immature adipocytes and mo...,myxoid liposarcoma with abundant ground substa...,open-where
3,/data/mn27889/path-open-data/pathvqa-histopath...,Where is presence of abundant coarse black car...,in the septal walls and around the bronchiole,open-where
4,/data/mn27889/path-open-data/pathvqa-histopath...,"Where is admixture of mature lymphocytes, plas...",in the centre of the field,open-where


Selecting all samples of When Open-Ended

In [24]:
pvqa_eval_data_open_when = pd.DataFrame(columns=['image_path', 'question', 'answer', 'question_type'])

for sample in when_question_samples:
    if 'train' in sample['image']:
        pvqa_images_subset_path = pvqa_images_subset_path_train
    elif 'test' in sample['image']:
        pvqa_images_subset_path = pvqa_images_subset_path_test
    else:
        pvqa_images_subset_path = pvqa_images_subset_path_val

    image_path = os.path.join(pvqa_images_subset_path, sample['image'] + '.jpg')
    question = sample['question']
    answer = sample['answer']
    question_type = 'open-when'
    pvqa_eval_data_open_when.loc[len(pvqa_eval_data_open_when)] = [image_path, question, answer, question_type]

pvqa_eval_data_open_when.head()

,image_path,question,answer,question_type
0,/data/mn27889/path-open-data/pathvqa-histopath...,When does periodic acid-Schiff stain ?,after diastase digestion of the liver,open-when


### Uploading all the images to Google Drive and get the drive links

Since we will be using the Google Form for the evaluation, we need to upload all the images to a specific Google Drive Folder. Then we need to get the drive link of each image and provide it to evaluators.

1. Move all the PathVQA images from server into a specific folder
2. Upload the Folder to google drive
3. Prepare a Google App Script to get the name and links (URL) of those files from the google drive folder in a google sheet
4. Map the names from Google Sheet and Dataframes to get the URLs of each image onto Google Drive
5. The resulting dataframes will be the final csv files which will be provided to evaluators

Firstly moving all the images for all question types in a specific folder to be uploaded to Google Driver

In [25]:
unique_images_path_all = np.concatenate([pvqa_eval_data_close_ended['image_path'].unique(),
                                        pvqa_eval_data_open_what['image_path'].unique(),
                                        pvqa_eval_data_open_how['image_path'].unique(),
                                        pvqa_eval_data_open_why['image_path'].unique(),
                                        pvqa_eval_data_open_where['image_path'].unique(),
                                        pvqa_eval_data_open_when['image_path'].unique()])
unique_images_path_all = np.unique(unique_images_path_all)
print('Total Unique Images for Evaluation:', len(unique_images_path_all))

Total Unique Images for Evaluation: 438


Moving all these images in a folder

In [26]:
pvqa_eval_images_dir = 'PathVQA_Eval_Images'
os.makedirs(pvqa_eval_images_dir, exist_ok=True)
for image_path in unique_images_path_all:
    shutil.copy(image_path, pvqa_eval_images_dir)

Now upload this folder onto the Google Driver. Then run the following scritpin Apps Script (script.google.com)

In [27]:
# function listFolderContents2() {
#   var foldername = 'PathVQA_Eval_Images';
#   var folderlisting = 'File Names and Links - '+ foldername;

#   var folders = DriveApp.getFoldersByName(foldername);
#   var folder = folders.next();
#   var contents = folder.getFiles();

#   var ss = SpreadsheetApp.create(folderlisting);
#   var sheet = ss.getActiveSheet();
#   sheet.appendRow(['name','link']);

#   var file;
#   var name;
#   var link;
#   var row;

#   while(contents.hasNext()) {
#     file = contents.next();
#     name = file.getName();
#     link = file.getUrl();
#     sheet.appendRow([name,link]);
#   }
# };

After running the above script, a new excel file will be created with the names and Google Drive Links of the files. That excel sheet needs to be downloaded and mapped back to all the individual question sets to finalize the image URLs in the Google Drive

In [28]:
data_eval_dir = 'data_eval'
pathvqa_drive_links_file = os.path.join(data_eval_dir, "file_names_and_links_PathVQA_Eval_Images.csv")
pathvqa_drive_links = pd.read_csv(pathvqa_drive_links_file)
pathvqa_drive_links.head()

,name,link
0,train_0060.jpg,https://drive.google.com/file/d/1-UhaEsCxL38Du...
1,train_0882.jpg,https://drive.google.com/file/d/1-Z5POwSdYlMbt...
2,train_0460.jpg,https://drive.google.com/file/d/1-kWVeCJBarnD3...
3,train_0017.jpg,https://drive.google.com/file/d/1-t3e5ljfqo51Q...
4,train_0201.jpg,https://drive.google.com/file/d/104iIpSOlXkFiD...


Changing the name of each file to complete path for correct mapping later on

In [29]:
pvqa_images_subset_path

'/data/mn27889/path-open-data/pathvqa-histopathology/images/train'

In [30]:
def extract_image_path(image_name):
    if 'train' in image_name:
        pvqa_images_subset_path = pvqa_images_subset_path_train
    elif 'test' in image_name:
        pvqa_images_subset_path = pvqa_images_subset_path_test
    else:
        pvqa_images_subset_path = pvqa_images_subset_path_val
    return os.path.join(pvqa_images_subset_path, image_name)

In [31]:
pathvqa_drive_links['image_path'] = pathvqa_drive_links['name'].apply(lambda x: extract_image_path(x))
pathvqa_drive_links['image_id'] = pathvqa_drive_links['name'].apply(lambda x: x.split('.')[0])
pathvqa_drive_links.head()

,name,link,image_path,image_id
0,train_0060.jpg,https://drive.google.com/file/d/1-UhaEsCxL38Du...,/data/mn27889/path-open-data/pathvqa-histopath...,train_0060
1,train_0882.jpg,https://drive.google.com/file/d/1-Z5POwSdYlMbt...,/data/mn27889/path-open-data/pathvqa-histopath...,train_0882
2,train_0460.jpg,https://drive.google.com/file/d/1-kWVeCJBarnD3...,/data/mn27889/path-open-data/pathvqa-histopath...,train_0460
3,train_0017.jpg,https://drive.google.com/file/d/1-t3e5ljfqo51Q...,/data/mn27889/path-open-data/pathvqa-histopath...,train_0017
4,train_0201.jpg,https://drive.google.com/file/d/104iIpSOlXkFiD...,/data/mn27889/path-open-data/pathvqa-histopath...,train_0201


### Mapping the Google Drive Links with each question set separately

In [32]:
pvqa_eval_data_close_ended = pd.merge(pvqa_eval_data_close_ended, pathvqa_drive_links, on='image_path', how='left')
pvqa_eval_data_close_ended.head()

,image_path,question,answer,question_type,name,link,image_id
0,/data/mn27889/path-open-data/pathvqa-histopath...,Are bile duct cells and canals of Hering stain...,yes,close-ended,train_0422.jpg,https://drive.google.com/file/d/1CC4p7nuihv490...,train_0422
1,/data/mn27889/path-open-data/pathvqa-histopath...,Are iron deposits shown by a special staining ...,yes,close-ended,train_0227.jpg,https://drive.google.com/file/d/1d165sE0faiQuc...,train_0227
2,/data/mn27889/path-open-data/pathvqa-histopath...,Do the photomicrographs show an inflammatory r...,yes,close-ended,train_0147.jpg,https://drive.google.com/file/d/1zSOb0qv8RYe2X...,train_0147
3,/data/mn27889/path-open-data/pathvqa-histopath...,Do apoptotic cells in colonic epithelium show ...,no,close-ended,train_0147.jpg,https://drive.google.com/file/d/1zSOb0qv8RYe2X...,train_0147
4,/data/mn27889/path-open-data/pathvqa-histopath...,Do the photomicrographs show an inflammatory r...,yes,close-ended,train_0157.jpg,https://drive.google.com/file/d/1xDcuhbI9tqpb5...,train_0157


In [33]:
pvqa_eval_data_open_what = pd.merge(pvqa_eval_data_open_what, pathvqa_drive_links, on='image_path', how='left')
pvqa_eval_data_open_what.head()

,image_path,question,answer,question_type,name,link,image_id
0,/data/mn27889/path-open-data/pathvqa-histopath...,What are stained here with an immunohistochemi...,bile duct cells and canals of hering,open-what,train_0422.jpg,https://drive.google.com/file/d/1CC4p7nuihv490...,train_0422
1,/data/mn27889/path-open-data/pathvqa-histopath...,What is the late-phase reaction characterized by?,an inflammatory infiltrate rich in eosinophils,open-what,train_0740.jpg,https://drive.google.com/file/d/1oVsG20uGgWMfr...,train_0740
2,/data/mn27889/path-open-data/pathvqa-histopath...,What is manifested by inflammatory cells in th...,acute cellular rejection of a kidney graft,open-what,train_0363.jpg,https://drive.google.com/file/d/1gJXgy1jsNhMjv...,train_0363
3,/data/mn27889/path-open-data/pathvqa-histopath...,What is acute cellular rejection of a kidney g...,inflammatory cells in the inter-stitium and be...,open-what,train_0363.jpg,https://drive.google.com/file/d/1gJXgy1jsNhMjv...,train_0363
4,/data/mn27889/path-open-data/pathvqa-histopath...,What reveals pink-red deposits of amyloid in t...,a section liver stained with congo,open-what,train_0344.jpg,https://drive.google.com/file/d/1TzPvmbLxep-V8...,train_0344


In [34]:
pvqa_eval_data_open_how = pd.merge(pvqa_eval_data_open_how, pathvqa_drive_links, on='image_path', how='left')
pvqa_eval_data_open_how.head()

,image_path,question,answer,question_type,name,link,image_id
0,/data/mn27889/path-open-data/pathvqa-histopath...,How is acute cellular rejection of a kidney gr...,by inflammatory cells in the inter-stitium and...,open-how,train_0363.jpg,https://drive.google.com/file/d/1gJXgy1jsNhMjv...,train_0363
1,/data/mn27889/path-open-data/pathvqa-histopath...,How is vascular changes and fibrosis of saliva...,by radiation therapy of the neck region,open-how,train_0969.jpg,https://drive.google.com/file/d/1vo50cLMWFncVI...,train_0969
2,/data/mn27889/path-open-data/pathvqa-histopath...,How is vascular changes and fibrosis of saliva...,by radiation therapy of the neck region,open-how,train_0911.jpg,https://drive.google.com/file/d/1NM3k2IB0Ou1-3...,train_0911
3,/data/mn27889/path-open-data/pathvqa-histopath...,How is vascular changes and fibrosis of saliva...,by radiation therapy of the neck region,open-how,train_0942.jpg,https://drive.google.com/file/d/1X43MdlGz3aOC2...,train_0942
4,/data/mn27889/path-open-data/pathvqa-histopath...,How is acute endocarditis caused?,by staphylococcus aureus on a congenitally bic...,open-how,train_0296.jpg,https://drive.google.com/file/d/1bsA171pggO0Pl...,train_0296


In [35]:
pvqa_eval_data_open_where = pd.merge(pvqa_eval_data_open_where, pathvqa_drive_links, on='image_path', how='left')
pvqa_eval_data_open_where.head()

,image_path,question,answer,question_type,name,link,image_id
0,/data/mn27889/path-open-data/pathvqa-histopath...,Where are liver stem cells (oval cells) located?,in the canals of hering,open-where,train_0422.jpg,https://drive.google.com/file/d/1CC4p7nuihv490...,train_0422
1,/data/mn27889/path-open-data/pathvqa-histopath...,Where does hepatocellular iron appear blue?,in the prussian blue-stained section,open-where,train_0456.jpg,https://drive.google.com/file/d/1yZ9oiCsN1obbV...,train_0456
2,/data/mn27889/path-open-data/pathvqa-histopath...,Where are scattered immature adipocytes and mo...,myxoid liposarcoma with abundant ground substa...,open-where,train_0589.jpg,https://drive.google.com/file/d/13JVGceOkmXrbi...,train_0589
3,/data/mn27889/path-open-data/pathvqa-histopath...,Where is presence of abundant coarse black car...,in the septal walls and around the bronchiole,open-where,train_0008.jpg,https://drive.google.com/file/d/1RISum1DiCwwdc...,train_0008
4,/data/mn27889/path-open-data/pathvqa-histopath...,"Where is admixture of mature lymphocytes, plas...",in the centre of the field,open-where,train_0891.jpg,https://drive.google.com/file/d/1krHoB-lpyIB1h...,train_0891


In [36]:
pvqa_eval_data_open_why = pd.merge(pvqa_eval_data_open_why, pathvqa_drive_links, on='image_path', how='left')
pvqa_eval_data_open_why.head()

,image_path,question,answer,question_type,name,link,image_id
0,/data/mn27889/path-open-data/pathvqa-histopath...,Why are the alveolar septa thickened?,due to congested capillaries and neutrophilic ...,open-why,train_0325.jpg,https://drive.google.com/file/d/1vBwyTIntB1_xM...,train_0325
1,/data/mn27889/path-open-data/pathvqa-histopath...,Why does this image show that cut surface both...,due to mumps have no history at time,open-why,train_2526.jpg,https://drive.google.com/file/d/1uOdG-P7EqjHzE...,train_2526
2,/data/mn27889/path-open-data/pathvqa-histopath...,"Why does this image show brain, infarct?",due to ruptured saccular aneurysm and thrombos...,open-why,train_1304.jpg,https://drive.google.com/file/d/18T89XymenJE-F...,train_1304
3,/data/mn27889/path-open-data/pathvqa-histopath...,"Why does this image show brain, infarct?",due to ruptured saccular aneurysm and thrombos...,open-why,train_1311.jpg,https://drive.google.com/file/d/1i0qSMtb5OE_vv...,train_1311
4,/data/mn27889/path-open-data/pathvqa-histopath...,Why does this image show spinal cord injury?,due to vertebral column trauma,open-why,train_1318.jpg,https://drive.google.com/file/d/1vIh8REvpc5ned...,train_1318


In [37]:
pvqa_eval_data_open_when = pd.merge(pvqa_eval_data_open_when, pathvqa_drive_links, on='image_path', how='left')
pvqa_eval_data_open_when.head()

,image_path,question,answer,question_type,name,link,image_id
0,/data/mn27889/path-open-data/pathvqa-histopath...,When does periodic acid-Schiff stain ?,after diastase digestion of the liver,open-when,train_0503.jpg,https://drive.google.com/file/d/1E9BkW-_ywhZ7V...,train_0503


### Creating the final dataset



Joining all these datasets into one dataframe to extract the final dataset to be used for evaluation of PathVQA

In [38]:
pvqa_eval_data_open_ended = pd.concat([pvqa_eval_data_open_what,
                                   pvqa_eval_data_open_how,
                                   pvqa_eval_data_open_why,
                                   pvqa_eval_data_open_where,
                                   pvqa_eval_data_open_when]).reset_index(drop=True)

pvqa_eval_data_open_ended.head()

,image_path,question,answer,question_type,name,link,image_id
0,/data/mn27889/path-open-data/pathvqa-histopath...,What are stained here with an immunohistochemi...,bile duct cells and canals of hering,open-what,train_0422.jpg,https://drive.google.com/file/d/1CC4p7nuihv490...,train_0422
1,/data/mn27889/path-open-data/pathvqa-histopath...,What is the late-phase reaction characterized by?,an inflammatory infiltrate rich in eosinophils,open-what,train_0740.jpg,https://drive.google.com/file/d/1oVsG20uGgWMfr...,train_0740
2,/data/mn27889/path-open-data/pathvqa-histopath...,What is manifested by inflammatory cells in th...,acute cellular rejection of a kidney graft,open-what,train_0363.jpg,https://drive.google.com/file/d/1gJXgy1jsNhMjv...,train_0363
3,/data/mn27889/path-open-data/pathvqa-histopath...,What is acute cellular rejection of a kidney g...,inflammatory cells in the inter-stitium and be...,open-what,train_0363.jpg,https://drive.google.com/file/d/1gJXgy1jsNhMjv...,train_0363
4,/data/mn27889/path-open-data/pathvqa-histopath...,What reveals pink-red deposits of amyloid in t...,a section liver stained with congo,open-what,train_0344.jpg,https://drive.google.com/file/d/1TzPvmbLxep-V8...,train_0344


### Suffling the dataframe with fixed seed

In [39]:
pvqa_eval_data_close_ended_shuffle = pvqa_eval_data_close_ended[['image_id', 'link', 'question_type', 'question', 'answer']].sample(frac=1, random_state=random_state).reset_index(drop=True)
pvqa_eval_data_close_ended_shuffle.head()

,image_id,link,question_type,question,answer
0,train_0476,https://drive.google.com/file/d/1kSXRzip_zfZOm...,close-ended,Does the remote kidney infarct show liquefacti...,no
1,train_0627,https://drive.google.com/file/d/1NyfVTxM2DXQGZ...,close-ended,Is the basement membrane intact?,yes
2,train_0627,https://drive.google.com/file/d/1NyfVTxM2DXQGZ...,close-ended,Is the entire thickness of the epithelium repl...,yes
3,train_0010,https://drive.google.com/file/d/1z3GwbOnYyLBCE...,close-ended,Is this tumor composed of small cells embedded...,yes
4,train_0363,https://drive.google.com/file/d/1gJXgy1jsNhMjv...,close-ended,Are t_h17 cells in granuloma formation outline...,no


In [40]:
pvqa_eval_data_open_ended_shuffle = pvqa_eval_data_open_ended[['image_id', 'link', 'question_type', 'question', 'answer']].sample(frac=1, random_state=random_state).reset_index(drop=True)
pvqa_eval_data_open_ended_shuffle.head()

,image_id,link,question_type,question,answer
0,train_0889,https://drive.google.com/file/d/1xX3d0GmNmvyg1...,open-what,What are seen as prussian blue granules?,haemosiderin pigment in the cytoplasm of hepat...
1,train_2683,https://drive.google.com/file/d/1RA4WCBvvZn1km...,open-what,What does this image show?,x-ray postmortcoronary angiogram rather typica...
2,train_0795,https://drive.google.com/file/d/1eRaM_mP0Y1za2...,open-what,What are lobular carcinomas composed of?,noncohesive tumor cells that invade as linear ...
3,train_0589,https://drive.google.com/file/d/13JVGceOkmXrbi...,open-where,Where are scattered immature adipocytes and mo...,myxoid liposarcoma with abundant ground substa...
4,train_1686,https://drive.google.com/file/d/1vIbd0TTKZtMmf...,open-what,What does this image show?,typical lesion rich in eosinophils source


#### Dividing the dataframe into five small dfs

In [42]:
df_part_len = int(len(pvqa_eval_data_close_ended_shuffle)/5)
pathopen_eval_data_shuffle_close_ended_part1 = pvqa_eval_data_close_ended_shuffle.iloc[0:df_part_len]
pathopen_eval_data_shuffle_close_ended_part2 = pvqa_eval_data_close_ended_shuffle.iloc[df_part_len:2*df_part_len]
pathopen_eval_data_shuffle_close_ended_part3 = pvqa_eval_data_close_ended_shuffle.iloc[2*df_part_len:3*df_part_len]
pathopen_eval_data_shuffle_close_ended_part4 = pvqa_eval_data_close_ended_shuffle.iloc[3*df_part_len:4*df_part_len]
pathopen_eval_data_shuffle_close_ended_part5 = pvqa_eval_data_close_ended_shuffle.iloc[4*df_part_len:len(pvqa_eval_data_close_ended_shuffle)]

In [43]:
df_part_len = int(len(pvqa_eval_data_open_ended_shuffle)/5)
pathopen_eval_data_shuffle_open_ended_part1 = pvqa_eval_data_open_ended_shuffle.iloc[0:df_part_len]
pathopen_eval_data_shuffle_open_ended_part2 = pvqa_eval_data_open_ended_shuffle.iloc[df_part_len:2*df_part_len]
pathopen_eval_data_shuffle_open_ended_part3 = pvqa_eval_data_open_ended_shuffle.iloc[2*df_part_len:3*df_part_len]
pathopen_eval_data_shuffle_open_ended_part4 = pvqa_eval_data_open_ended_shuffle.iloc[3*df_part_len:4*df_part_len]
pathopen_eval_data_shuffle_open_ended_part5 = pvqa_eval_data_open_ended_shuffle.iloc[4*df_part_len:len(pvqa_eval_data_open_ended_shuffle)]

#### Combining Close and Open Ended Dataframes for different parts to form the final evaluation dataset for pathologists

In [44]:
pathopen_eval_data_shuffle_part1 = pd.concat([pathopen_eval_data_shuffle_close_ended_part1, pathopen_eval_data_shuffle_open_ended_part1])
pathopen_eval_data_shuffle_part2 = pd.concat([pathopen_eval_data_shuffle_close_ended_part2, pathopen_eval_data_shuffle_open_ended_part2])
pathopen_eval_data_shuffle_part3 = pd.concat([pathopen_eval_data_shuffle_close_ended_part3, pathopen_eval_data_shuffle_open_ended_part3])
pathopen_eval_data_shuffle_part4 = pd.concat([pathopen_eval_data_shuffle_close_ended_part4, pathopen_eval_data_shuffle_open_ended_part4])
pathopen_eval_data_shuffle_part5 = pd.concat([pathopen_eval_data_shuffle_close_ended_part5, pathopen_eval_data_shuffle_open_ended_part5])

In [45]:
pathopen_eval_data_shuffle_part1.to_csv('data_eval/pathvqa_part1.csv', index=False)
pathopen_eval_data_shuffle_part2.to_csv('data_eval/pathvqa_part2.csv', index=False)
pathopen_eval_data_shuffle_part3.to_csv('data_eval/pathvqa_part3.csv', index=False)
pathopen_eval_data_shuffle_part4.to_csv('data_eval/pathvqa_part4.csv', index=False)
pathopen_eval_data_shuffle_part5.to_csv('data_eval/pathvqa_part5.csv', index=False)